In [2]:
# Bibliotecas
import os
import json
import sqlite3
import pandas as pd 
import time

from sqlalchemy import create_engine
import urllib.parse
from dotenv import load_dotenv

In [2]:
# Configuração do Pipeline
DB_PATH = 'hardware_humano.db'
BASE_DIR = './osfstorage-archive/pmdata'
USERS_RANGE = range(1, 17)

In [3]:
# Função para conecão do banco de dados
def load_data():
    # 1. Conecta ao banco de dados SQLite
    # Usamos um gerenciador de contexto para garantir que a conexão seja fechada adequadamente
    try:
        conn = sqlite3.connect(DB_PATH)
        print("Conexão com o banco de dados estabelecida")
    except sqlite3.Error as e:
        print(f"Erro ao conectar ao banco de dados: {e}")
        return

    # 2. Percorre as pastas de usuários (p1 a 16)
    for user_id in USERS_RANGE:
        folder_name = f'p{user_id}'
        folder_path = os.path.join(BASE_DIR, folder_name)

        # Verifica se a pasta do usuário existe
        if not os.path.isdir(folder_path):
            print(f"Pasta não encontrada, pulando: {folder_name}") 
            continue

        print(f" Processando dados do usuário, {user_id} ({folder_name})...") 

        # Coluna: Heart Rate (JSON)
        hr_file = os.path.join(folder_path, 'fitbit', 'heart_rate.json')
        if os.path.exists(hr_file):
            try:
                # Lê o JSON com pandas
                df_hr = pd.read_json(hr_file)
                df_hr['userId'] = user_id
                
                # Tratamento da coluna tempo
                if 'dateTime' in df_hr.columns:
                    df_hr['timestamp'] = df_hr['dateTime']
                elif 'time' in df_hr.columns:
                    df_hr['timestamp'] = df_hr['time']

                # Tratamento da coluna BPM
                if 'value' in df_hr.columns:
                    # O código abaixo verifica se é um diconário e extra apena o número do BPM
                    if isinstance(df_hr['value'].dropna().iloc[0], dict):
                        df_hr['bpm'] = df_hr['value'].apply(lambda x: x.get('bpm') if isinstance(x, dict) else x)
                    else:
                        df_hr = df_hr.rename(columns={'value': 'bpm'})

                colunas_hr = ['userId', 'timestamp', 'bpm']

                # Validação de segurança
                colunas_faltantes = [col for col in colunas_hr if col not in df_hr.columns]
                if colunas_faltantes:
                    print(f"Alerta: Colunas {colunas_faltantes} não encontradas em {folder_name}/heart_rate. Colunas atuais: {list(df_hr.columns)}")
                else:
                    df_hr[colunas_hr].to_sql('heart_rate', conn, if_exists='append', index=False)
                    print(f"Sucesso: heart_rate.json ({len(df_hr)} registros)")

            except Exception as e:
                print(f"Erro ao processar {hr_file}: {e}") 


        # Coluna: SLEEP (JSON)
        sleep_file = os.path.join(folder_path, 'fitbit', 'sleep.json')
        if os.path.exists(sleep_file):
            try:
                df_sleep = pd.read_json(sleep_file)
                df_sleep['userId'] = user_id

                # 1. Correção de mapeamento: usando 'startTime' em vez de 'dateOfSleep'
                df_sleep = df_sleep.rename(columns={
                    'startTime': 'start_time', 
                    'endTime': 'end_time',
                    'minutesAsleep': 'minutes_asleep',
                    'efficiency': 'sleep_score'
                })

                # Colunas exigidas pelo banco de dados
                colunas_sleep = ['userId', 'start_time', 'end_time', 'minutes_asleep', 'sleep_score']

                # 2. Prevenção de KeyError: Garante que todas as colunas existam no DF
                for col in colunas_sleep:
                    if col not in df_sleep.columns:
                        df_sleep[col] = None # Preenche com NULL no SQLite caso a métrica falte

                # Carga segura no banco
                df_sleep[colunas_sleep].to_sql('sleep', conn, if_exists='append', index=False)
                print(f"Sucesso: sleep.json ({len(df_sleep)} registros)")
                
            except Exception as e:
                print(f"Erro ao processar {sleep_file}: {e}")

        # Arquivo: Well Being (CSV)
        wb_file = os.path.join(folder_path, 'pmsys', 'wellness.csv')
        if not os.path.exists(wb_file):
            print(f"Aviso: Arquivo não encontrado no caminho -> {wb_file}")
        else:
            try:
                # Lendo o cs
                df_wb = pd.read_csv(wb_file)

                # Padronizando as colunas
                df_wb.columns = df_wb.columns.str.lower()

                # Adicionando o userId para a grafia exata exigida pelo banco
                df_wb['userId'] = user_id

                # Mapeando a coluna de data usando o novo nome da coluna descoberta
                if 'effective_time_frame' in df_wb.columns:
                    df_wb['date'] = df_wb['effective_time_frame']
                elif 'effective_time' in df_wb.columns:
                    df_wb['date'] = df_wb['effective_time']
                elif 'time' in df_wb.columns:
                    df_wb['date'] = df_wb['time']
                
                # As colunas que queremos extrair e salvar
                colunas_wb = ['userId', 'date', 'fatigue', 'mood', 'readiness', 'stress']

                # Validação de segurança
                colunas_faltantes = [col for col in colunas_wb if col not in df_wb.columns]

                if colunas_faltantes:
                    print(f" Erro: colunas {colunas_faltantes} faltando no CSV. Colunas atuais: {list(df_wb.columns)}")
                else:
                    df_wb[colunas_wb].to_sql('well_being', conn, if_exists='append', index=False)
                    print(f" Sucesso: wellness.csv ({len(df_wb)} registros)")

            except Exception as e:
                print(f"Erro ao processar {wb_file}: {e}")

        # 3. Fecha a conexão
    conn.close()
    print("Pipeline de ETL concluído com sucesso")

if __name__ == "__main__":
    load_data()            

Conexão com o banco de dados estabelecida
 Processando dados do usuário, 1 (p1)...
Sucesso: heart_rate.json (1573165 registros)
Sucesso: sleep.json (155 registros)
 Sucesso: wellness.csv (138 registros)
 Processando dados do usuário, 2 (p2)...
Sucesso: heart_rate.json (1472629 registros)
Sucesso: sleep.json (158 registros)
 Sucesso: wellness.csv (99 registros)
 Processando dados do usuário, 3 (p3)...
Sucesso: heart_rate.json (808341 registros)
Sucesso: sleep.json (84 registros)
 Sucesso: wellness.csv (82 registros)
 Processando dados do usuário, 4 (p4)...
Sucesso: heart_rate.json (1571315 registros)
Sucesso: sleep.json (188 registros)
 Sucesso: wellness.csv (145 registros)
 Processando dados do usuário, 5 (p5)...
Sucesso: heart_rate.json (1370967 registros)
Sucesso: sleep.json (133 registros)
 Sucesso: wellness.csv (137 registros)
 Processando dados do usuário, 6 (p6)...
Sucesso: heart_rate.json (1579882 registros)
Sucesso: sleep.json (165 registros)
 Sucesso: wellness.csv (147 registr

In [4]:
# Conecta ao banco
conn = sqlite3.connect('hardware_humano.db')
cursor = conn.cursor()

# Verificando quais tabelas realmente existem no arquivo
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tabelas_existentes = cursor.fetchall()
print(f"Tabelas encontradas no banco: {[t[0] for t in tabelas_existentes]}")

# garantindo que a tebela_being exista
cursor.execute("""
CREATE TABLE IF NOT EXISTS well_being (
    userId INTEGER,
    date TEXT,
    fatigue INTEGER,
    mood INTEGER,
    readiness INTEGER,
    stress INTEGER
);
""")
conn.commit()
print("Estrutura da tabela 'well_being' garantida.")

Tabelas encontradas no banco: ['sleep', 'heart_rate', 'well_being']
Estrutura da tabela 'well_being' garantida.


In [5]:
# Query
query = """
SELECT 
    base.userId,
    (SELECT COUNT(*) FROM heart_rate WHERE userId = base.userId) AS total_heart_rate,
    (SELECT COUNT(*) FROM sleep WHERE userId = base.userId) AS total_sleep,
    (SELECT COUNT(*) FROM well_being WHERE userId = base.userId) AS total_well_being
FROM (
    SELECT userId FROM heart_rate
    UNION
    SELECT userId FROM sleep
    UNION
    SELECT userId FROM well_being
) AS base
ORDER BY base.userId;
"""

# Executa e mostra o resultado em uma tabela 
df_verificacao = pd.read_sql_query(query, conn)
conn.close()

# Exibe o DataFrame
df_verificacao

,userId,total_heart_rate,total_sleep,total_well_being
0,1,18877980,2170,966
1,2,17671548,2212,693
2,3,9700092,1176,574
3,4,18855780,2632,1015
4,5,15080637,1729,959
5,6,17378702,2145,1029
6,7,17401417,2093,931
7,8,17746586,1859,728
8,9,14360720,1846,714
9,10,11915827,1442,672


In [3]:
# Carrega as variáveis do arquivo oculto .env
load_dotenv()

# Puxando as credenciais com segurança
db_user = os.getenv("DB_USER")
db_password = urllib.parse.quote_plus(os.getenv("DB_PASSWORD"))
db_host = os.getenv("DB_HOST")
db_port = os.getenv("DB_PORT")
db_name = os.getenv("DB_NAME")

# Monta a URI corrigida (Corrigido: db_port)
POSTGRES_URI = f'postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}'
SQLITE_PATH = os.getenv("SQLITE_PATH")

In [7]:
def migrar_dados():
    # Criando engines de conexão
    sqlite_conn = sqlite3.connect(SQLITE_PATH)
    pg_engine = create_engine(POSTGRES_URI)

    tabelas = ['heart_rate', 'sleep', 'well_being']

    print("Iniciando migração dos dados...")
    for tabela in tabelas:
        print(f"Transferindo tabela: {tabela}...")
        chunk_size = 100000 # chunk_size para não travar a tabela heart_rate
        
        # Lendo e escrevendo os blocos
        query = f"SELECT * FROM {tabela}"
        for chunk in pd.read_sql_query(query, sqlite_conn, chunksize=chunk_size):
            # Garantindo conversão de datas
            if 'timestamp' in chunk.columns:
                chunk['timestamp'] = pd.to_datetime(chunk['timestamp'])
            if 'start_time' in chunk.columns:
                chunk['start_time'] = pd.to_datetime(chunk['start_time'])
            if 'end_time' in chunk.columns:
                chunk['end_time'] = pd.to_datetime(chunk['end_time'])
            if 'date' in chunk.columns:
                chunk['date'] = pd.to_datetime(chunk['date'])
                
            # Insere no Postgres
            chunk.to_sql(tabela, pg_engine, if_exists='append', index=False)
            print(f"+{len(chunk)} linhas inseridas no Postgres...")

    # Fechando a conexão do loop
    sqlite_conn.close()
    print("Migração concluída com sucesso para o PostgreSQL!")

if __name__ == "__main__":
    migrar_dados()

Iniciando migração dos dados...
Transferindo tabela: heart_rate...
+100000 linhas inseridas no Postgres...
+100000 linhas inseridas no Postgres...
+100000 linhas inseridas no Postgres...
+100000 linhas inseridas no Postgres...
+100000 linhas inseridas no Postgres...
+100000 linhas inseridas no Postgres...
+100000 linhas inseridas no Postgres...
+100000 linhas inseridas no Postgres...
+100000 linhas inseridas no Postgres...
+100000 linhas inseridas no Postgres...
+100000 linhas inseridas no Postgres...
+100000 linhas inseridas no Postgres...
+100000 linhas inseridas no Postgres...
+100000 linhas inseridas no Postgres...
+100000 linhas inseridas no Postgres...
+100000 linhas inseridas no Postgres...
+100000 linhas inseridas no Postgres...
+100000 linhas inseridas no Postgres...
+100000 linhas inseridas no Postgres...
+100000 linhas inseridas no Postgres...
+100000 linhas inseridas no Postgres...
+100000 linhas inseridas no Postgres...
+100000 linhas inseridas no Postgres...
+100000 linha